# Gh?p video t? audio segments c? s?n

Notebook n?y ch? ch?y b??c sync/gh?p video. Ch?y l?n l??t:

1. Upload file zip audio c? c?u tr?c gi?ng `omnivoice_audio_results.zip` trong ?nh, v? d? m?i t?p c? folder ri?ng v? b?n trong c? `segments/` + `*_grouped.srt`.
2. Upload video g?c t??ng ?ng (`1.mp4`, `2.mp4`, ...).
3. Ch?y cell sync ?? t?o `dichvideo_final_videos.zip`, g?m video gh?p th?nh c?ng v? `batch_sync_errors.log` ?? xem t?p n?o l?i.


In [ ]:
from google.colab import files
from pathlib import Path
import json
import re
import shutil
import sys
import traceback
import zipfile

AUDIO_UPLOAD_DIR = Path('/content/dichvideo_audio_zip_uploads')
AUDIO_ROOT = Path('/content/dichvideo_omnivoice_audio')
VIDEO_DIR = Path('/content/dichvideo_video_uploads')
SYNC_JOBS_DIR = Path('/content/dichvideo_sync_jobs')
FINAL_VIDEO_DIR = Path('/content/dichvideo_final_videos')

for folder in [AUDIO_UPLOAD_DIR, AUDIO_ROOT, SYNC_JOBS_DIR, FINAL_VIDEO_DIR]:
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir(parents=True, exist_ok=True)
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_EXTENSIONS = {'.mp4', '.mov', '.mkv', '.webm', '.avi'}
AUDIO_EXTENSIONS = {'.wav', '.mp3', '.m4a', '.flac', '.ogg'}

print('Upload audio results .zip, v? d?: omnivoice_audio_results.zip')
uploaded_audio = files.upload()

zip_paths = []
for name, data in uploaded_audio.items():
    path = Path('/content') / name
    path.write_bytes(data)
    if path.suffix.lower() == '.zip':
        zip_paths.append(path)
    else:
        print(f'Skipped non-zip file: {name}')

if not zip_paths:
    raise RuntimeError('Upload at least one audio results .zip file.')

for zip_path in zip_paths:
    print('Extracting:', zip_path.name)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(AUDIO_ROOT)
    shutil.move(str(zip_path), AUDIO_UPLOAD_DIR / zip_path.name)

print('Audio job folders found:')
audio_job_folders = sorted(path for path in AUDIO_ROOT.rglob('*') if path.is_dir() and (path / 'segments').is_dir())
for folder in audio_job_folders:
    print('-', folder.relative_to(AUDIO_ROOT))
if not audio_job_folders:
    raise RuntimeError(f'No audio job folders with segments/ found in {AUDIO_ROOT}')


In [ ]:
from google.colab import files
from pathlib import Path
import shutil

VIDEO_DIR = Path('/content/dichvideo_video_uploads')
VIDEO_EXTENSIONS = {'.mp4', '.mov', '.mkv', '.webm', '.avi'}

shutil.rmtree(VIDEO_DIR, ignore_errors=True)
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

print('Upload original video file(s), v? d?: 1.mp4, 2.mp4, 3.mp4...')
uploaded_videos = files.upload()

video_paths = []
for name, data in uploaded_videos.items():
    suffix = Path(name).suffix.lower()
    if suffix in VIDEO_EXTENSIONS:
        target = VIDEO_DIR / name
        target.write_bytes(data)
        video_paths.append(target)
    else:
        print(f'Skipped non-video file: {name}')

if not video_paths:
    raise RuntimeError('Upload at least one original video file.')

print('Uploaded videos:')
for path in sorted(VIDEO_DIR.iterdir()):
    print('-', path.name)


In [ ]:
# Embedded local Sync SRT + Audio Segments helper from dichvideo/segment_retimer.py
from pathlib import Path

PACKAGE_DIR = Path('/content/dichvideo')
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
(PACKAGE_DIR / '__init__.py').write_text('', encoding='utf-8')
(PACKAGE_DIR / 'segment_retimer.py').write_text('from __future__ import annotations\n\nimport json\nimport logging\nimport re\nimport shutil\nimport subprocess\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Iterable\n\n\n@dataclass\nclass SegmentRetimeConfig:\n    min_video_ratio: float = 0.70\n    max_video_ratio: float = 1.40\n    min_audio_tempo: float = 0.75\n    max_audio_tempo: float = 1.80\n    keep_gaps: bool = True\n    original_audio_volume: float = 0.0\n    crf: int = 20\n    preset: str = "veryfast"\n    output_fps: int = 30\n\n\n@dataclass\nclass SegmentRetimeUpdate:\n    status: str\n    log_text: str\n    video_path: str | None = None\n    schedule_path: str | None = None\n    log_path: str | None = None\n    job_path: str | None = None\n\n\n@dataclass\nclass VideoTimelinePiece:\n    start: float\n    end: float\n    ratio: float\n    duration: float\n\n\ndef retime_video_to_audio_segments(\n    video_path: Path,\n    srt_path: Path,\n    audio_paths: list[Path],\n    jobs_root: Path,\n    config: SegmentRetimeConfig,\n) -> Iterable[SegmentRetimeUpdate]:\n    job_dir = _create_job_dir(jobs_root)\n    log_path = job_dir / "logs" / "segment_retimer.log"\n    logger = _setup_logger(log_path)\n\n    def update(message: str, video: Path | None = None, schedule: Path | None = None) -> SegmentRetimeUpdate:\n        logger.info(message)\n        return SegmentRetimeUpdate(\n            status=message,\n            log_text=_tail(log_path),\n            video_path=str(video) if video and video.exists() else None,\n            schedule_path=str(schedule) if schedule and schedule.exists() else None,\n            log_path=str(log_path),\n            job_path=str(job_dir),\n        )\n\n    try:\n        _check_binary("ffmpeg")\n        _check_binary("ffprobe")\n        if not video_path.exists():\n            raise RuntimeError(f"Video not found: {video_path}")\n        if not srt_path.exists():\n            raise RuntimeError(f"SRT not found: {srt_path}")\n        if not audio_paths:\n            raise RuntimeError("No audio segment files provided.")\n\n        input_video = job_dir / "input" / video_path.name\n        input_srt = job_dir / "input" / srt_path.name\n        shutil.copy2(video_path, input_video)\n        shutil.copy2(srt_path, input_srt)\n        copied_audio_paths = []\n        for index, audio_path in enumerate(_sort_audio_paths(audio_paths), start=1):\n            copied = job_dir / "input" / "audio_segments" / f"{index:04d}{audio_path.suffix.lower() or \'.wav\'}"\n            copied.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(audio_path, copied)\n            copied_audio_paths.append(copied)\n\n        yield update(f"Created segment retime job: {job_dir}")\n\n        segments = parse_srt(input_srt.read_text(encoding="utf-8-sig"))\n        if len(copied_audio_paths) < len(segments):\n            raise RuntimeError(f"Need at least {len(segments)} audio segments, got {len(copied_audio_paths)}.")\n        if len(copied_audio_paths) > len(segments):\n            logger.warning("More audio files than SRT segments. Extra files will be ignored: %s > %s", len(copied_audio_paths), len(segments))\n            copied_audio_paths = copied_audio_paths[:len(segments)]\n\n        video_duration = _duration(input_video, logger)\n        _validate_segments(segments, video_duration)\n        has_original_audio = config.original_audio_volume > 0 and _has_audio(input_video, logger)\n        if config.original_audio_volume > 0 and not has_original_audio:\n            logger.warning("Original audio volume was requested, but source video has no readable audio stream.")\n        yield update(f"Loaded {len(segments)} SRT segment(s). Video duration: {video_duration:.3f}s")\n\n        video_timeline = []\n        audio_piece_list = []\n        original_audio_piece_list = []\n        schedule = []\n        cursor = 0.0\n        previous_end = 0.0\n\n        for index, (segment, audio_path) in enumerate(zip(segments, copied_audio_paths), start=1):\n            if config.keep_gaps and segment["start"] - previous_end >= 0.05:\n                gap_duration = segment["start"] - previous_end\n                gap_audio = job_dir / "work" / "audio_pieces" / f"{len(audio_piece_list):05d}_gap.wav"\n                video_timeline.append(VideoTimelinePiece(previous_end, segment["start"], 1.0, gap_duration))\n                _make_silence(gap_audio, gap_duration, logger)\n                audio_piece_list.append(gap_audio)\n                if has_original_audio:\n                    original_gap_audio = job_dir / "work" / "original_audio_pieces" / f"{len(original_audio_piece_list):05d}_gap.wav"\n                    _extract_original_audio_piece(input_video, previous_end, segment["start"], 1.0, gap_duration, original_gap_audio, logger)\n                    original_audio_piece_list.append(original_gap_audio)\n                schedule.append({\n                    "type": "gap",\n                    "original_start": round(previous_end, 3),\n                    "original_end": round(segment["start"], 3),\n                    "output_start": round(cursor, 3),\n                    "output_end": round(cursor + gap_duration, 3),\n                    "duration": round(gap_duration, 3),\n                })\n                cursor += gap_duration\n\n            original_video_duration = max(0.001, segment["end"] - segment["start"])\n            original_audio_duration = _duration(audio_path, logger)\n            desired_video_ratio = original_audio_duration / original_video_duration\n            video_ratio = _clamp(desired_video_ratio, config.min_video_ratio, config.max_video_ratio)\n            target_duration = original_video_duration * video_ratio\n\n            if abs(target_duration - original_audio_duration) <= 0.03:\n                audio_tempo = 1.0\n            else:\n                audio_tempo = _clamp(original_audio_duration / target_duration, config.min_audio_tempo, config.max_audio_tempo)\n\n            audio_piece = job_dir / "work" / "audio_pieces" / f"{len(audio_piece_list):05d}_seg_{index:04d}.wav"\n            video_timeline.append(VideoTimelinePiece(segment["start"], segment["end"], video_ratio, target_duration))\n            _retime_audio_piece(audio_path, audio_tempo, target_duration, audio_piece, logger)\n            actual_audio_piece_duration = _duration(audio_piece, logger)\n            if abs(target_duration - actual_audio_piece_duration) > 0.08:\n                logger.warning(\n                    "Rendered segment duration mismatch index=%s target=%.3fs video_piece=%.3fs audio_piece=%.3fs",\n                    index,\n                    target_duration,\n                    target_duration,\n                    actual_audio_piece_duration,\n                )\n            if has_original_audio:\n                original_audio_piece = job_dir / "work" / "original_audio_pieces" / f"{len(original_audio_piece_list):05d}_seg_{index:04d}.wav"\n                _extract_original_audio_piece(\n                    input_video,\n                    segment["start"],\n                    segment["end"],\n                    1.0 / video_ratio,\n                    target_duration,\n                    original_audio_piece,\n                    logger,\n                )\n                original_audio_piece_list.append(original_audio_piece)\n\n            audio_piece_list.append(audio_piece)\n            schedule.append({\n                "type": "segment",\n                "index": segment["index"],\n                "text": segment["text"],\n                "original_start": round(segment["start"], 3),\n                "original_end": round(segment["end"], 3),\n                "original_video_duration": round(original_video_duration, 3),\n                "original_audio_duration": round(original_audio_duration, 3),\n                "desired_video_ratio": round(desired_video_ratio, 5),\n                "video_ratio": round(video_ratio, 5),\n                "target_duration": round(target_duration, 3),\n                "audio_tempo": round(audio_tempo, 5),\n                "actual_video_piece_duration": round(target_duration, 3),\n                "actual_audio_piece_duration": round(actual_audio_piece_duration, 3),\n                "output_start": round(cursor, 3),\n                "output_end": round(cursor + target_duration, 3),\n                "audio_file": str(audio_path.name),\n            })\n            logger.info(\n                "Segment %s video=%.3fs audio=%.3fs desired_ratio=%.5f video_ratio=%.5f target=%.3fs audio_tempo=%.5f",\n                index,\n                original_video_duration,\n                original_audio_duration,\n                desired_video_ratio,\n                video_ratio,\n                target_duration,\n                audio_tempo,\n            )\n            cursor += target_duration\n            previous_end = segment["end"]\n            yield update(f"Processed segment {index}/{len(segments)}")\n\n        if config.keep_gaps and video_duration - previous_end >= 0.05:\n            gap_duration = video_duration - previous_end\n            gap_audio = job_dir / "work" / "audio_pieces" / f"{len(audio_piece_list):05d}_tail.wav"\n            video_timeline.append(VideoTimelinePiece(previous_end, video_duration, 1.0, gap_duration))\n            _make_silence(gap_audio, gap_duration, logger)\n            audio_piece_list.append(gap_audio)\n            if has_original_audio:\n                original_tail_audio = job_dir / "work" / "original_audio_pieces" / f"{len(original_audio_piece_list):05d}_tail.wav"\n                _extract_original_audio_piece(input_video, previous_end, video_duration, 1.0, gap_duration, original_tail_audio, logger)\n                original_audio_piece_list.append(original_tail_audio)\n            schedule.append({\n                "type": "tail",\n                "original_start": round(previous_end, 3),\n                "original_end": round(video_duration, 3),\n                "output_start": round(cursor, 3),\n                "output_end": round(cursor + gap_duration, 3),\n                "duration": round(gap_duration, 3),\n            })\n            cursor += gap_duration\n\n        schedule_path = job_dir / "output" / "retime_schedule.json"\n        _write_json(schedule_path, {\n            "source_video": str(input_video),\n            "source_srt": str(input_srt),\n            "config": config.__dict__,\n            "output_duration": round(cursor, 3),\n            "items": schedule,\n        })\n\n        yield update("Rendering retimed video timeline and concatenating audio...")\n        retimed_video = job_dir / "output" / "retimed_video.mp4"\n        retimed_audio = job_dir / "output" / "retimed_audio.wav"\n        _render_video_timeline(input_video, video_timeline, retimed_video, config, logger, job_dir)\n        _concat_media(audio_piece_list, retimed_audio, "audio", logger, job_dir)\n        audio_for_mux = retimed_audio\n        if has_original_audio:\n            original_retimed_audio = job_dir / "output" / "original_retimed_audio.wav"\n            mixed_audio = job_dir / "output" / "mixed_audio.wav"\n            _concat_media(original_audio_piece_list, original_retimed_audio, "original_audio", logger, job_dir)\n            _mix_audio(original_retimed_audio, retimed_audio, mixed_audio, config.original_audio_volume, logger)\n            audio_for_mux = mixed_audio\n\n        final_video = job_dir / "output" / "final.mp4"\n        _run([\n            "ffmpeg", "-y",\n            "-i", str(retimed_video),\n            "-i", str(audio_for_mux),\n            "-map", "0:v:0",\n            "-map", "1:a:0",\n            "-c:v", "copy",\n            "-c:a", "aac",\n            "-shortest",\n            str(final_video),\n        ], logger)\n        yield update("Done", video=final_video, schedule=schedule_path)\n    except Exception:\n        logger.exception("Segment retime failed")\n        yield SegmentRetimeUpdate(\n            status="Segment retime failed. Xem log de biet chi tiet.",\n            log_text=_tail(log_path),\n            log_path=str(log_path),\n            job_path=str(job_dir),\n        )\n\n\ndef parse_srt(content: str) -> list[dict]:\n    content = content.replace("\\r\\n", "\\n").replace("\\r", "\\n").strip()\n    if not content:\n        return []\n    blocks = re.split(r"\\n\\s*\\n", content)\n    segments = []\n    fallback_index = 1\n    for block in blocks:\n        lines = [line.strip() for line in block.split("\\n") if line.strip()]\n        if not lines:\n            continue\n        timing_line_index = next((i for i, line in enumerate(lines) if "-->" in line), None)\n        if timing_line_index is None:\n            continue\n        maybe_index = lines[0] if timing_line_index > 0 else str(fallback_index)\n        index = int(re.sub(r"\\D+", "", maybe_index) or fallback_index)\n        timing = lines[timing_line_index]\n        start_s, end_s = [part.strip().split()[0] for part in timing.split("-->", 1)]\n        text = " ".join(lines[timing_line_index + 1:]).strip()\n        if text:\n            segments.append({\n                "index": index,\n                "start": _parse_srt_timestamp(start_s),\n                "end": _parse_srt_timestamp(end_s),\n                "text": text,\n            })\n            fallback_index += 1\n    return segments\n\n\ndef _parse_srt_timestamp(value: str) -> float:\n    match = re.match(r"(\\d+):(\\d+):(\\d+)[,.](\\d+)", value)\n    if not match:\n        raise ValueError(f"Invalid SRT timestamp: {value}")\n    h, m, s, ms = match.groups()\n    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms.ljust(3, "0")[:3]) / 1000\n\n\ndef _extract_video_piece(input_video: Path, start: float, end: float, output_path: Path, config: SegmentRetimeConfig, logger: logging.Logger) -> None:\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    duration = max(0.001, end - start)\n    video_filter = (\n        f"trim=start={start:.6f}:end={end:.6f},"\n        "setpts=PTS-STARTPTS,"\n        f"fps={config.output_fps},"\n        "tpad=stop_mode=clone:stop_duration=1,"\n        f"trim=duration={duration:.6f},"\n        "setpts=PTS-STARTPTS"\n    )\n    _run([\n        "ffmpeg", "-y",\n        "-i", str(input_video),\n        "-an",\n        "-filter:v", video_filter,\n        "-c:v", "libx264",\n        "-preset", config.preset,\n        "-crf", str(config.crf),\n        "-pix_fmt", "yuv420p",\n        str(output_path),\n    ], logger)\n\n\ndef _extract_and_retime_video_piece(input_video: Path, start: float, end: float, ratio: float, target_duration: float, output_path: Path, config: SegmentRetimeConfig, logger: logging.Logger) -> None:\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    target_duration = max(0.001, target_duration)\n    video_filter = (\n        f"trim=start={start:.6f}:end={end:.6f},"\n        "setpts=PTS-STARTPTS,"\n        f"setpts={ratio:.8f}*PTS,"\n        f"fps={config.output_fps},"\n        "tpad=stop_mode=clone:stop_duration=1,"\n        f"trim=duration={target_duration:.6f},"\n        "setpts=PTS-STARTPTS"\n    )\n    _run([\n        "ffmpeg", "-y",\n        "-i", str(input_video),\n        "-an",\n        "-filter:v", video_filter,\n        "-c:v", "libx264",\n        "-preset", config.preset,\n        "-crf", str(config.crf),\n        "-pix_fmt", "yuv420p",\n        str(output_path),\n    ], logger)\n\n\ndef _render_video_timeline(\n    input_video: Path,\n    pieces: list[VideoTimelinePiece],\n    output_path: Path,\n    config: SegmentRetimeConfig,\n    logger: logging.Logger,\n    job_dir: Path,\n) -> None:\n    if not pieces:\n        raise RuntimeError("No video timeline pieces to render.")\n\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    filter_script_path = job_dir / "work" / "video_timeline_filter.txt"\n    filter_script_path.parent.mkdir(parents=True, exist_ok=True)\n\n    lines = []\n    labels = []\n    for index, piece in enumerate(pieces):\n        duration = max(0.001, piece.duration)\n        label = f"v{index}"\n        labels.append(f"[{label}]")\n        lines.append(\n            f"[0:v]trim=start={piece.start:.6f}:end={piece.end:.6f},"\n            "setpts=PTS-STARTPTS,"\n            f"setpts={piece.ratio:.8f}*PTS,"\n            f"fps={config.output_fps},"\n            "tpad=stop_mode=clone:stop_duration=1,"\n            f"trim=duration={duration:.6f},"\n            f"setpts=PTS-STARTPTS[{label}]"\n        )\n\n    lines.append("".join(labels) + f"concat=n={len(pieces)}:v=1:a=0[vout]")\n    filter_script_path.write_text(";\\n".join(lines), encoding="utf-8")\n\n    _run([\n        "ffmpeg", "-y",\n        "-i", str(input_video),\n        "-filter_complex_script", str(filter_script_path),\n        "-map", "[vout]",\n        "-an",\n        "-c:v", "libx264",\n        "-preset", config.preset,\n        "-crf", str(config.crf),\n        "-pix_fmt", "yuv420p",\n        str(output_path),\n    ], logger)\n\n\ndef _retime_audio_piece(input_audio: Path, tempo: float, target_duration: float, output_path: Path, logger: logging.Logger) -> None:\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    filters = []\n    if abs(tempo - 1.0) > 0.001:\n        filters.append(_atempo_filter(tempo))\n    filters.extend(["apad", f"atrim=0:{target_duration:.3f}"])\n    _run([\n        "ffmpeg", "-y",\n        "-i", str(input_audio),\n        "-filter:a", ",".join(filters),\n        "-ac", "2",\n        "-ar", "44100",\n        str(output_path),\n    ], logger)\n\n\ndef _extract_original_audio_piece(input_video: Path, start: float, end: float, tempo: float, target_duration: float, output_path: Path, logger: logging.Logger) -> None:\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    target_duration = max(0.001, target_duration)\n    filters = [f"atrim=start={start:.6f}:end={end:.6f}", "asetpts=PTS-STARTPTS"]\n    if abs(tempo - 1.0) > 0.001:\n        filters.append(_atempo_filter(tempo))\n    filters.extend(["apad", f"atrim=0:{target_duration:.6f}", "asetpts=PTS-STARTPTS"])\n    _run([\n        "ffmpeg", "-y",\n        "-i", str(input_video),\n        "-vn",\n        "-filter:a", ",".join(filters),\n        "-ac", "2",\n        "-ar", "44100",\n        str(output_path),\n    ], logger)\n\n\ndef _make_silence(output_path: Path, duration: float, logger: logging.Logger) -> None:\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    _run([\n        "ffmpeg", "-y",\n        "-f", "lavfi",\n        "-i", "anullsrc=channel_layout=stereo:sample_rate=44100",\n        "-t", f"{duration:.3f}",\n        str(output_path),\n    ], logger)\n\n\ndef _concat_media(paths: list[Path], output_path: Path, kind: str, logger: logging.Logger, job_dir: Path) -> None:\n    list_path = job_dir / "work" / f"{kind}_concat.txt"\n    list_path.parent.mkdir(parents=True, exist_ok=True)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    list_path.write_text("".join(f"file \'{path.resolve().as_posix()}\'\\n" for path in paths), encoding="utf-8")\n    if kind == "video":\n        cmd = ["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(list_path), "-c:v", "libx264", "-preset", "veryfast", "-crf", "20", "-pix_fmt", "yuv420p", str(output_path)]\n    else:\n        cmd = ["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(list_path), "-c:a", "pcm_s16le", "-ac", "2", "-ar", "44100", str(output_path)]\n    _run(cmd, logger)\n\n\ndef _mix_audio(original_audio: Path, dubbed_audio: Path, output_path: Path, original_volume: float, logger: logging.Logger) -> None:\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    volume = max(0.0, float(original_volume))\n    _run([\n        "ffmpeg", "-y",\n        "-i", str(original_audio),\n        "-i", str(dubbed_audio),\n        "-filter_complex", f"[0:a]volume={volume:.5f}[a0];[a0][1:a]amix=inputs=2:normalize=0[out]",\n        "-map", "[out]",\n        "-ac", "2",\n        "-ar", "44100",\n        str(output_path),\n    ], logger)\n\n\ndef _atempo_filter(tempo: float) -> str:\n    parts = []\n    remaining = tempo\n    while remaining > 2.0:\n        parts.append("atempo=2.0")\n        remaining /= 2.0\n    while remaining < 0.5:\n        parts.append("atempo=0.5")\n        remaining /= 0.5\n    parts.append(f"atempo={remaining:.5f}")\n    return ",".join(parts)\n\n\ndef _sort_audio_paths(paths: list[Path]) -> list[Path]:\n    def key(path: Path):\n        numbers = re.findall(r"\\d+", path.stem)\n        return (int(numbers[-1]) if numbers else 10**9, path.name.lower())\n    return sorted(paths, key=key)\n\n\ndef _validate_segments(segments: list[dict], video_duration: float) -> None:\n    previous_end = 0.0\n    for index, segment in enumerate(segments, start=1):\n        start = float(segment["start"])\n        end = float(segment["end"])\n        if end <= start:\n            raise RuntimeError(\n                f"SRT segment {index} has invalid timing: start={_format_seconds(start)}, end={_format_seconds(end)}."\n            )\n        if start < previous_end - 0.001:\n            raise RuntimeError(\n                f"SRT segment {index} starts before the previous segment ends: "\n                f"previous_end={_format_seconds(previous_end)}, start={_format_seconds(start)}."\n            )\n        if end > video_duration + 1.0:\n            raise RuntimeError(\n                f"SRT segment {index} ends after the video duration: "\n                f"end={_format_seconds(end)}, video_duration={_format_seconds(video_duration)}."\n            )\n        previous_end = max(previous_end, end)\n\n\ndef _format_seconds(value: float) -> str:\n    return f"{value:.3f}s"\n\n\ndef _duration(path: Path, logger: logging.Logger) -> float:\n    completed = _run([\n        "ffprobe", "-v", "error",\n        "-show_entries", "format=duration",\n        "-of", "default=noprint_wrappers=1:nokey=1",\n        str(path),\n    ], logger)\n    return float(completed.stdout.strip())\n\n\ndef _has_audio(path: Path, logger: logging.Logger) -> bool:\n    completed = _run([\n        "ffprobe", "-v", "error",\n        "-select_streams", "a:0",\n        "-show_entries", "stream=index",\n        "-of", "csv=p=0",\n        str(path),\n    ], logger)\n    return bool(completed.stdout.strip())\n\n\ndef _clamp(value: float, minimum: float, maximum: float) -> float:\n    return max(minimum, min(maximum, value))\n\n\ndef _create_job_dir(jobs_root: Path) -> Path:\n    stamp = time.strftime("%Y%m%d_%H%M%S")\n    jobs_root.mkdir(parents=True, exist_ok=True)\n    for counter in range(1000):\n        suffix = "" if counter == 0 else f"_{counter:03d}"\n        job_dir = jobs_root / f"segment_retime_{stamp}{suffix}"\n        try:\n            job_dir.mkdir(parents=True, exist_ok=False)\n            for child in ["input", "work", "output", "logs"]:\n                (job_dir / child).mkdir(parents=True, exist_ok=True)\n            return job_dir\n        except FileExistsError:\n            continue\n    raise RuntimeError(f"Could not create unique segment retime job folder under: {jobs_root}")\n\n\ndef _setup_logger(log_path: Path) -> logging.Logger:\n    logger = logging.getLogger(f"dichvideo.segment_retimer.{log_path.parent.parent.name}")\n    logger.setLevel(logging.INFO)\n    logger.handlers.clear()\n    formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")\n    file_handler = logging.FileHandler(log_path, encoding="utf-8")\n    file_handler.setFormatter(formatter)\n    logger.addHandler(file_handler)\n    stream_handler = logging.StreamHandler()\n    stream_handler.setFormatter(formatter)\n    logger.addHandler(stream_handler)\n    return logger\n\n\ndef _run(cmd: list[str], logger: logging.Logger) -> subprocess.CompletedProcess:\n    logger.info("Running command: %s", " ".join(cmd))\n    completed = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")\n    if completed.stdout.strip():\n        logger.info("stdout: %s", completed.stdout.strip()[-3000:])\n    if completed.stderr.strip():\n        logger.info("stderr: %s", completed.stderr.strip()[-3000:])\n    if completed.returncode != 0:\n        raise RuntimeError(f"Command failed with code {completed.returncode}: {\' \'.join(cmd)}")\n    return completed\n\n\ndef _check_binary(name: str) -> None:\n    if shutil.which(name) is None:\n        raise RuntimeError(f"Missing dependency: {name}. Hay cai ffmpeg va them vao PATH.")\n\n\ndef _write_json(path: Path, data) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")\n\n\ndef _tail(path: Path, lines: int = 160) -> str:\n    if not path.exists():\n        return ""\n    return "\\n".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:])\n', encoding='utf-8')


In [ ]:
from pathlib import Path
import re
import shutil
import sys
import traceback

sys.path.insert(0, '/content')

from dichvideo.segment_retimer import SegmentRetimeConfig, retime_video_to_audio_segments

# Sync settings. SRT ???c ph?p d?i h?n video t?i ?a kho?ng 1 gi?y trong helper retimer.
MIN_VIDEO_RATIO = 0.25
MAX_VIDEO_RATIO = 3.0
MIN_AUDIO_TEMPO = 0.75
MAX_AUDIO_TEMPO = 1.8
KEEP_GAPS = True
ORIGINAL_AUDIO_VOLUME = 0.0
CRF = 20
PRESET = 'veryfast'
OUTPUT_FPS = 30

VIDEO_EXTENSIONS = {'.mp4', '.mov', '.mkv', '.webm', '.avi'}
AUDIO_EXTENSIONS = {'.wav', '.mp3', '.m4a', '.flac', '.ogg'}

def safe_name(value: str) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', value).strip('._') or 'item'

def match_key(path_or_name) -> str:
    stem = Path(path_or_name).stem if not isinstance(path_or_name, str) else Path(path_or_name).stem
    return safe_name(stem).lower().replace('__', '_')

def video_number(video_path: Path):
    match = re.fullmatch(r'(\d+)', video_path.stem.strip())
    return int(match.group(1)) if match else None

def folder_episode_number(folder: Path) -> int:
    name = folder.name.lower()
    stem = re.sub(r'_(vi|vn)$', '', name, flags=re.IGNORECASE)
    if stem.isdigit():
        return int(stem)
    match = re.search(r'(?:^|[_\-\(])(\d+)[\)]?$', stem)
    return int(match.group(1)) + 1 if match else 1

def find_grouped_srt(job_dir: Path) -> Path:
    grouped = sorted(job_dir.glob('*_grouped.srt'))
    if grouped:
        return grouped[0]
    srt_files = sorted(job_dir.glob('*.srt'))
    if srt_files:
        return srt_files[0]
    raise RuntimeError(f'No .srt file found in audio job folder: {job_dir}')

def discover_audio_jobs(audio_root: Path) -> list[Path]:
    jobs = []
    for folder in sorted(path for path in audio_root.rglob('*') if path.is_dir()):
        segments_dir = folder / 'segments'
        if not segments_dir.is_dir():
            continue
        if any(path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS for path in segments_dir.iterdir()):
            jobs.append(folder)
    return jobs

def match_video_for_job(job_dir: Path, video_files: list[Path]) -> Path:
    if len(video_files) == 1:
        return video_files[0]

    episode = folder_episode_number(job_dir)
    numbered_videos = {number: video for video in video_files if (number := video_number(video)) is not None}
    if episode in numbered_videos:
        return numbered_videos[episode]

    job_key = match_key(job_dir.name)
    by_key = {match_key(video): video for video in video_files}
    if job_key in by_key:
        return by_key[job_key]
    candidates = [video for key, video in by_key.items() if job_key in key or key in job_key]
    if len(candidates) == 1:
        return candidates[0]

    raise RuntimeError(
        f'Cannot match video for audio folder {job_dir.name}. '
        f'D?ng t?n video d?ng 1.mp4, 2.mp4... ho?c upload ch? 1 video.'
    )

video_files = sorted(path for path in VIDEO_DIR.iterdir() if path.suffix.lower() in VIDEO_EXTENSIONS)
audio_jobs = discover_audio_jobs(AUDIO_ROOT)

if not video_files:
    raise RuntimeError(f'No video files found in {VIDEO_DIR}')
if not audio_jobs:
    raise RuntimeError(f'No audio job folders with segments/ found in {AUDIO_ROOT}')

config = SegmentRetimeConfig(
    min_video_ratio=float(MIN_VIDEO_RATIO),
    max_video_ratio=float(MAX_VIDEO_RATIO),
    min_audio_tempo=float(MIN_AUDIO_TEMPO),
    max_audio_tempo=float(MAX_AUDIO_TEMPO),
    keep_gaps=bool(KEEP_GAPS),
    original_audio_volume=float(ORIGINAL_AUDIO_VOLUME),
    crf=int(CRF),
    preset=str(PRESET),
    output_fps=int(OUTPUT_FPS),
)

shutil.rmtree(FINAL_VIDEO_DIR, ignore_errors=True)
FINAL_VIDEO_DIR.mkdir(parents=True, exist_ok=True)
final_paths = []
failed_items = []
batch_log_path = FINAL_VIDEO_DIR / 'batch_sync_errors.log'

for job_dir in audio_jobs:
    video_path = None
    sync_srt_path = None
    last_update = None
    try:
        video_path = match_video_for_job(job_dir, video_files)
        sync_srt_path = find_grouped_srt(job_dir)
        audio_dir = job_dir / 'segments'
        audio_paths = sorted(
            [path for path in audio_dir.iterdir() if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS],
            key=lambda p: (int(re.findall(r'\d+', p.stem)[-1]) if re.findall(r'\d+', p.stem) else 10**9, p.name.lower())
        )
        if not audio_paths:
            raise RuntimeError(f'No audio segment files found: {audio_dir}')

        print(f'Syncing {video_path.name} + {sync_srt_path.name} + {len(audio_paths)} audio segment(s)...')
        for update in retime_video_to_audio_segments(video_path, sync_srt_path, audio_paths, SYNC_JOBS_DIR, config):
            last_update = update
            print(update.status)
        if not last_update or not last_update.video_path:
            raise RuntimeError(f'Sync failed. Log: {last_update.log_path if last_update else "unknown"}')

        final_name = f'{safe_name(job_dir.name)}_final.mp4'
        final_path = FINAL_VIDEO_DIR / final_name
        shutil.copy2(last_update.video_path, final_path)
        if last_update.schedule_path:
            shutil.copy2(last_update.schedule_path, FINAL_VIDEO_DIR / f'{safe_name(job_dir.name)}_retime_schedule.json')
        if last_update.log_path:
            shutil.copy2(last_update.log_path, FINAL_VIDEO_DIR / f'{safe_name(job_dir.name)}_segment_retimer.log')
        final_paths.append(final_path)
        print('Final video:', final_path)
    except Exception as exc:
        log_copy_path = None
        if last_update and last_update.log_path and Path(last_update.log_path).exists():
            log_copy_path = FINAL_VIDEO_DIR / f'{safe_name(job_dir.name)}_FAILED_segment_retimer.log'
            shutil.copy2(last_update.log_path, log_copy_path)
        failed_items.append({
            'audio_folder': str(job_dir.relative_to(AUDIO_ROOT)),
            'video': video_path.name if video_path else 'unknown',
            'sync_srt': sync_srt_path.name if sync_srt_path else 'unknown',
            'error': str(exc),
            'traceback': traceback.format_exc(),
            'retimer_log': str(log_copy_path) if log_copy_path else (last_update.log_path if last_update else 'unknown'),
        })
        print(f'Skipped {job_dir.name}: {exc}')

log_lines = [
    'DichVideo video-only sync log',
    f'Success: {len(final_paths)}',
    f'Failed: {len(failed_items)}',
    '',
]
if failed_items:
    for index, item in enumerate(failed_items, start=1):
        log_lines.extend([
            f'[{index}] Audio folder: {item["audio_folder"]}',
            f'    Video: {item["video"]}',
            f'    Sync SRT: {item["sync_srt"]}',
            f'    Error: {item["error"]}',
            '    Traceback:',
            item['traceback'].rstrip(),
            f'    Retimer log: {item["retimer_log"]}',
            '',
        ])
else:
    log_lines.append('No sync errors.')
batch_log_path.write_text('\n'.join(log_lines), encoding='utf-8')
print('Batch sync log:', batch_log_path)

zip_base = FINAL_VIDEO_DIR.parent / 'dichvideo_final_videos'
if zip_base.with_suffix('.zip').exists():
    zip_base.with_suffix('.zip').unlink()
shutil.make_archive(str(zip_base), 'zip', FINAL_VIDEO_DIR)
print('Created final videos zip:', zip_base.with_suffix('.zip'))


In [ ]:
from google.colab import files
from pathlib import Path

zip_path = Path('/content/dichvideo_final_videos.zip')
if zip_path.exists():
    files.download(str(zip_path))
else:
    print('No final video zip found.')
